# Gold — fact_defunciones

Tabla de hechos de defunciones. Fuentes: INE (5 tablas Silver) + WHO (2 tablas Silver).

| Columna | Tipo | Notas |
|---|---|---|
| `tiempo_sk` | INT | FK dim_tiempo |
| `geografia_sk` | INT | FK dim_geografia |
| `causa_sk` | INT | FK dim_causa |
| `demografia_sk` | INT | FK dim_demografia |
| `fuente_sk` | INT | FK dim_fuente |
| `total_defunciones` | BIGINT | INE: total / WHO: defunciones |
| `defunciones_hombres` | BIGINT | INE: hombres — NULL para WHO |
| `defunciones_mujeres` | BIGINT | INE: mujeres — NULL para WHO |
| `tasa_mortalidad_x100k` | DOUBLE | Solo WHO |
| `tasa_estandarizada_x100k` | DOUBLE | Solo WHO |

**ADVERTENCIA DE DOBLE CONTEO:** INE-edad (sin depto) e INE-geo (con depto) son cortes distintos
de las mismas defunciones. Filtrar por `dim_fuente.source_system` al analizar.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import LongType, DoubleType
from functools import reduce
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SILVER_SCHEMA = 'stage_silver_ss2'
GOLD_SCHEMA   = 'gold_ss2'
WRITE_MODE    = 'overwrite'

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return 'manual'

RUN_ID = get_job_run_id()
logger.info(f'Setup OK — run_id={RUN_ID}')

## Carga de dimensiones

In [0]:
dim_tiempo     = spark.table(f'{GOLD_SCHEMA}.dim_tiempo')
dim_geografia  = spark.table(f'{GOLD_SCHEMA}.dim_geografia')
dim_causa      = spark.table(f'{GOLD_SCHEMA}.dim_causa')
dim_demografia = spark.table(f'{GOLD_SCHEMA}.dim_demografia')
dim_fuente     = spark.table(f'{GOLD_SCHEMA}.dim_fuente')

# Buscar surrogate keys fijos para filas placeholder
geo_sk_sin_desagregar = (
    dim_geografia
    .filter((F.col('pais') == 'Guatemala') & (F.col('departamento') == 'Sin desagregar'))
    .select('geografia_sk')
    .first()['geografia_sk']
)
demo_sk_sin_desagregar = (
    dim_demografia
    .filter((F.col('sexo') == 'Ambos') & (F.col('grupo_etario_label') == 'Sin desagregar'))
    .select('demografia_sk')
    .first()['demografia_sk']
)

logger.info(f'Placeholder geo_sk (Sin desagregar): {geo_sk_sin_desagregar}')
logger.info(f'Placeholder demo_sk (Sin desagregar): {demo_sk_sin_desagregar}')

# Helpers de join reutilizables
def join_tiempo(df):
    return df.join(dim_tiempo.select('tiempo_sk', 'anio'), on='anio', how='left')

def join_causa_cie10(df, cie10_col):
    dim_c = dim_causa.select(F.col('causa_sk'), F.col('codigo_origen').alias('_c'))
    return df.join(dim_c, F.col(cie10_col) == F.col('_c'), how='left').drop('_c')

def join_causa_codigo(df, codigo_col):
    dim_c = dim_causa.select(F.col('causa_sk'), F.col('codigo_origen').alias('_c'))
    return df.join(dim_c, F.col(codigo_col) == F.col('_c'), how='left').drop('_c')

def join_fuente(df):
    dim_f = dim_fuente.select(
        F.col('fuente_sk'),
        F.col('source_system').alias('_fs'),
        F.col('source_file').alias('_ff')
    )
    return df.join(
        dim_f,
        (F.col('source_system') == F.col('_fs')) & (F.col('source_file') == F.col('_ff')),
        how='left'
    ).drop('_fs', '_ff')

def join_demografia_sexo_grupo(df, sexo_col, grupo_col):
    dim_d = dim_demografia.select(
        F.col('demografia_sk'),
        F.col('sexo').alias('_ds'),
        F.col('grupo_etario_label').alias('_dg')
    )
    return df.join(
        dim_d,
        (F.col(sexo_col) == F.col('_ds')) & (F.col(grupo_col) == F.col('_dg')),
        how='left'
    ).drop('_ds', '_dg')

def join_geo_depto(df, depto_col):
    dim_g = (
        dim_geografia
        .filter(F.col('nivel_geo') == 'departamento')
        .select(F.col('geografia_sk'), F.col('departamento').alias('_gd'))
    )
    return df.join(dim_g, F.col(depto_col) == F.col('_gd'), how='left').drop('_gd')

logger.info('Helpers de join listos')

## Fuente 1 — INE-edad (3 tablas)

Sin columna geografica -> `geo_sk` = placeholder (Guatemala / Sin desagregar).
`sexo='Ambos'` (hombres/mujeres son medidas, no dimension).
Join demografico por etiqueta de edad.

In [0]:
_INE_EDAD = [
    'ine_defunciones_sexo_edad',
    'ine_defunciones_neonatales',
    'ine_defunciones_postneonatales',
]

df_ine_edad_raw = reduce(lambda a, b: a.union(b), [
    spark.table(f'{SILVER_SCHEMA}.{t}').select(
        'anio', 'cie_10_norm', 'total', 'hombres', 'mujeres',
        'source_system', 'source_file',
        F.col('edad').alias('grupo_etario_label')
    )
    for t in _INE_EDAD
])

# Join tiempo
df = join_tiempo(df_ine_edad_raw)

# Geografia: placeholder fijo
df = df.withColumn('geografia_sk', F.lit(geo_sk_sin_desagregar))

# Causa
df = join_causa_cie10(df, 'cie_10_norm')

# Demografia: sexo=Ambos, grupo=edad col
dim_d = dim_demografia.select(
    F.col('demografia_sk'),
    F.col('sexo').alias('_ds'),
    F.col('grupo_etario_label').alias('_dg')
)
df = df.join(
    dim_d,
    (F.lit('Ambos') == F.col('_ds')) & (F.col('grupo_etario_label') == F.col('_dg')),
    how='left'
).drop('_ds', '_dg')

# Fuente
df = join_fuente(df)

df_ine_edad = df.select(
    'tiempo_sk', 'geografia_sk', 'causa_sk', 'demografia_sk', 'fuente_sk',
    F.col('total').cast(LongType()).alias('total_defunciones'),
    F.col('hombres').cast(LongType()).alias('defunciones_hombres'),
    F.col('mujeres').cast(LongType()).alias('defunciones_mujeres'),
    F.lit(None).cast(DoubleType()).alias('tasa_mortalidad_x100k'),
    F.lit(None).cast(DoubleType()).alias('tasa_estandarizada_x100k')
)
logger.info(f'INE-edad: {df_ine_edad.count():,} filas')

## Fuente 2 — INE-geo residencia

Join geografico por `departamento_oficial` (nivel departamento).
Sin desagregacion etaria -> `demo_sk` = placeholder (Ambos / Sin desagregar).

In [0]:
df_geo_res_raw = spark.table(f'{SILVER_SCHEMA}.ine_defunciones_depto_residencia').select(
    'anio', 'cie_10_norm', 'total', 'hombres', 'mujeres',
    'source_system', 'source_file', 'departamento_oficial'
)

df = join_tiempo(df_geo_res_raw)
df = join_geo_depto(df, 'departamento_oficial')
df = join_causa_cie10(df, 'cie_10_norm')
df = df.withColumn('demografia_sk', F.lit(demo_sk_sin_desagregar))
df = join_fuente(df)

df_ine_geo_res = df.select(
    'tiempo_sk', 'geografia_sk', 'causa_sk', 'demografia_sk', 'fuente_sk',
    F.col('total').cast(LongType()).alias('total_defunciones'),
    F.col('hombres').cast(LongType()).alias('defunciones_hombres'),
    F.col('mujeres').cast(LongType()).alias('defunciones_mujeres'),
    F.lit(None).cast(DoubleType()).alias('tasa_mortalidad_x100k'),
    F.lit(None).cast(DoubleType()).alias('tasa_estandarizada_x100k')
)
logger.info(f'INE-geo residencia: {df_ine_geo_res.count():,} filas')

## Fuente 3 — INE causas externas

Codigo CIE-10: columna `cie_10_norm` (rango normalizado, ej. `AccidentesdetransporteV01-V99`).
Join geografico por `departamento_oficial`.
Sin desagregacion etaria -> `demo_sk` = placeholder.

In [0]:
df_causas_ext_raw = spark.table(f'{SILVER_SCHEMA}.ine_defunciones_causas_externas').select(
    'anio', 'cie_10_norm', 'total', 'hombres', 'mujeres',
    'source_system', 'source_file', 'departamento_oficial'
)

df = join_tiempo(df_causas_ext_raw)
df = join_geo_depto(df, 'departamento_oficial')
df = join_causa_cie10(df, 'cie_10_norm')
df = df.withColumn('demografia_sk', F.lit(demo_sk_sin_desagregar))
df = join_fuente(df)

df_ine_causas_ext = df.select(
    'tiempo_sk', 'geografia_sk', 'causa_sk', 'demografia_sk', 'fuente_sk',
    F.col('total').cast(LongType()).alias('total_defunciones'),
    F.col('hombres').cast(LongType()).alias('defunciones_hombres'),
    F.col('mujeres').cast(LongType()).alias('defunciones_mujeres'),
    F.lit(None).cast(DoubleType()).alias('tasa_mortalidad_x100k'),
    F.lit(None).cast(DoubleType()).alias('tasa_estandarizada_x100k')
)
logger.info(f'INE causas externas: {df_ine_causas_ext.count():,} filas')

## Fuente 4 — WHO (Guatemala + Costa Rica)

Join geografico por `pais` (nivel pais, departamento IS NULL).
Join demografico por `(sexo, grupo_etario)`.
Hombres/mujeres son NULL (WHO no desagrega por sexo en las medidas de conteo).
Tasas solo disponibles en WHO.

In [0]:
df_who_raw = reduce(lambda a, b: a.union(b), [
    spark.table(f'{SILVER_SCHEMA}.{t}').select(
        'anio', 'codigo_causa', 'defunciones',
        'tasa_mortalidad_x100k', 'tasa_estandarizada_x100k',
        'pais', 'sexo', 'grupo_etario',
        'source_system', 'source_file'
    )
    for t in ['who_guatemala', 'who_costa_rica']
])

# Normalizar pais: Silver tiene 'guatemala'/'costa_rica', dim tiene 'Guatemala'/'Costa Rica'
df_who_raw = df_who_raw.withColumn(
    'pais', F.initcap(F.regexp_replace(F.col('pais'), '_', ' '))
)

# Join tiempo
df = join_tiempo(df_who_raw)

# Join geografia por pais (nivel='pais', departamento IS NULL)
dim_geo_pais = (
    dim_geografia
    .filter((F.col('nivel_geo') == 'pais') & F.col('departamento').isNull())
    .select(F.col('geografia_sk'), F.col('pais').alias('_gp'))
)
df = df.join(dim_geo_pais, F.col('pais') == F.col('_gp'), how='left').drop('_gp')

# Join causa por codigo_causa (indicador OMS)
df = join_causa_codigo(df, 'codigo_causa')

# Join demografia por (sexo, grupo_etario)
df = join_demografia_sexo_grupo(df, 'sexo', 'grupo_etario')

# Join fuente
df = join_fuente(df)

df_who = df.select(
    'tiempo_sk', 'geografia_sk', 'causa_sk', 'demografia_sk', 'fuente_sk',
    F.col('defunciones').alias('total_defunciones'),
    F.lit(None).cast(LongType()).alias('defunciones_hombres'),
    F.lit(None).cast(LongType()).alias('defunciones_mujeres'),
    'tasa_mortalidad_x100k',
    'tasa_estandarizada_x100k'
)
logger.info(f'WHO: {df_who.count():,} filas')

## UNION y escritura

In [0]:
_FACT_COLS = [
    'tiempo_sk', 'geografia_sk', 'causa_sk', 'demografia_sk', 'fuente_sk',
    'total_defunciones', 'defunciones_hombres', 'defunciones_mujeres',
    'tasa_mortalidad_x100k', 'tasa_estandarizada_x100k'
]

df_fact = reduce(lambda a, b: a.union(b), [
    df_ine_edad.select(*_FACT_COLS),
    df_ine_geo_res.select(*_FACT_COLS),
    df_ine_causas_ext.select(*_FACT_COLS),
    df_who.select(*_FACT_COLS),
])

total = df_fact.count()
logger.info(f'fact_defunciones total: {total:,} filas')

df_fact.write \
    .format('delta') \
    .mode(WRITE_MODE) \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD_SCHEMA}.fact_defunciones')

logger.info(f'Escrito -> {GOLD_SCHEMA}.fact_defunciones')

## Validacion

In [0]:
df_val = spark.table(f'{GOLD_SCHEMA}.fact_defunciones')
print(f'Total filas: {df_val.count():,}')

print('\n-- NULLs en claves foraneas --')
for col in ['tiempo_sk', 'geografia_sk', 'causa_sk', 'demografia_sk', 'fuente_sk']:
    n = df_val.filter(F.col(col).isNull()).count()
    print(f'  {col}: {n:,} NULLs')

print('\n-- Distribucion por fuente --')
(
    df_val
    .join(dim_fuente.select('fuente_sk', 'pipeline_name'), 'fuente_sk')
    .groupBy('pipeline_name')
    .agg(
        F.count('*').alias('filas'),
        F.sum('total_defunciones').alias('total_def')
    )
    .show()
)

print('\n-- Query demo: defunciones por anio y pais (WHO) --')
(
    df_val
    .join(dim_fuente.select('fuente_sk', 'pipeline_name'), 'fuente_sk')
    .filter(F.col('pipeline_name') == 'stage_who')
    .join(dim_tiempo.select('tiempo_sk', 'anio'), 'tiempo_sk')
    .join(
        dim_geografia.select('geografia_sk', 'pais'),
        'geografia_sk'
    )
    .groupBy('anio', 'pais')
    .agg(F.sum('total_defunciones').alias('defunciones'))
    .orderBy('pais', 'anio')
    .limit(20)
    .show()
)